# Custom YOLO Training Notebook (Colab)

This notebook is a cleaned, configuration-driven pipeline for training custom YOLO models with reliable resume support.

What it does:
- installs dependencies
- mounts Google Drive
- prepares dataset and updates `data.yaml`
- resumes interrupted training from the latest checkpoint when available
- exports `best.pt`, `last.pt`, and a zipped run folder to Drive
- runs validation and prediction using dynamic model paths (no hardcoded run paths)


In [ ]:
# Core setup
import glob
import os
import shutil
import zipfile
from datetime import datetime
from pathlib import Path

import pandas as pd
import torch
import yaml
from IPython.display import Image, display
from google.colab import drive

# Install Ultralytics in Colab runtime
!pip -q install --upgrade ultralytics

from ultralytics import YOLO


## 1) Global Configuration

Edit only this cell to reuse the notebook across datasets/projects.

`RUNS_ROOT` is intentionally on Google Drive so `last.pt` persists across runtime disconnects.


In [ ]:
# Paths
DATASET_ZIP = "/content/drive/MyDrive/datasets/Keypoint_reduced.zip"
DATASET_ROOT = "/content/datasets"
DRIVE_EXPORT_DIR = "/content/drive/MyDrive/yolo_exports"
RUNS_ROOT = "/content/drive/MyDrive/yolo_runs"

# Training config
MODEL_NAME = "yolo11n-pose"
PROJECT_NAME = "640-yolo11n-pose-mosaic"
EPOCHS = 100
IMG_SIZE = 640
BATCH_SIZE = 6
PATIENCE = 50
WORKERS = 8
CONF_PRED = 0.25

# Optional behavior
FORCE_REEXTRACT_DATASET = False

# Task inference from model name
if "pose" in MODEL_NAME:
    TASK = "pose"
elif "seg" in MODEL_NAME:
    TASK = "segment"
else:
    TASK = "detect"

# Device selection (GPU if available, otherwise CPU)
DEVICE = "0" if torch.cuda.is_available() else "cpu"

# Derived paths
DATASET_NAME = Path(DATASET_ZIP).stem
DATASET_DIR = os.path.join(DATASET_ROOT, DATASET_NAME)
YAML_PATH = os.path.join(DATASET_DIR, "data.yaml")
PROJECT_RUNS_DIR = os.path.join(RUNS_ROOT, TASK, PROJECT_NAME)

print("TASK:", TASK)
print("DEVICE:", DEVICE)
print("DATASET_DIR:", DATASET_DIR)
print("YAML_PATH:", YAML_PATH)
print("PROJECT_RUNS_DIR:", PROJECT_RUNS_DIR)


## 2) Mount Drive and Prepare Dataset

- mounts Drive
- extracts dataset zip
- rewrites `data.yaml` so `path` points to extracted dataset root
- verifies train/val/test paths


In [ ]:
# Mount Google Drive (required for dataset + persistent checkpoints)
drive.mount("/content/drive", force_remount=False)

os.makedirs(DATASET_ROOT, exist_ok=True)
os.makedirs(DRIVE_EXPORT_DIR, exist_ok=True)
os.makedirs(PROJECT_RUNS_DIR, exist_ok=True)

if FORCE_REEXTRACT_DATASET and os.path.exists(DATASET_DIR):
    shutil.rmtree(DATASET_DIR)

if not os.path.exists(DATASET_DIR):
    print("Extracting dataset zip:", DATASET_ZIP)
    with zipfile.ZipFile(DATASET_ZIP, "r") as zf:
        zf.extractall(DATASET_ROOT)
else:
    print("Dataset already extracted:", DATASET_DIR)

if not os.path.exists(YAML_PATH):
    raise FileNotFoundError(f"data.yaml not found at {YAML_PATH}")

with open(YAML_PATH, "r", encoding="utf-8") as f:
    data_cfg = yaml.safe_load(f)

data_cfg["path"] = DATASET_DIR

with open(YAML_PATH, "w", encoding="utf-8") as f:
    yaml.safe_dump(data_cfg, f, sort_keys=False)

print("Updated data.yaml path ->", data_cfg["path"])


def _resolve_split_path(root, split_value):
    if split_value is None:
        return None
    return split_value if os.path.isabs(split_value) else os.path.join(root, split_value)

for split in ["train", "val", "test"]:
    if split in data_cfg:
        p = _resolve_split_path(DATASET_DIR, data_cfg.get(split))
        print(f"{split}: {p} | exists={os.path.exists(p)}")


## 3) Train With Robust Resume

Resume behavior:
- first tries latest `last.pt` from persistent project runs in Drive
- if none found, tries exported `*-last.pt` checkpoints from `DRIVE_EXPORT_DIR`
- if checkpoint is mid-training: resumes
- if checkpoint is already finished: starts a new run initialized from that checkpoint
- if no checkpoint exists: starts from base model


In [ ]:
def latest_file(pattern):
    files = glob.glob(pattern)
    return max(files, key=os.path.getmtime) if files else None

# Prefer persistent run checkpoints first
resume_ckpt = latest_file(os.path.join(PROJECT_RUNS_DIR, "*", "weights", "last.pt"))

# Fallback to exported last checkpoints
if resume_ckpt is None:
    resume_ckpt = latest_file(os.path.join(DRIVE_EXPORT_DIR, f"{PROJECT_NAME}-*-last.pt"))

run_stamp = datetime.now().strftime("%Y-%m-%d-%H%M%S")
run_name = f"{PROJECT_NAME}_{run_stamp}"

train_args = {
    "data": YAML_PATH,
    "epochs": EPOCHS,
    "imgsz": IMG_SIZE,
    "batch": BATCH_SIZE,
    "patience": PATIENCE,
    "workers": WORKERS,
    "device": DEVICE,
    "project": PROJECT_RUNS_DIR,
    "name": run_name,
    "exist_ok": False,
    "plots": True,
}

if resume_ckpt:
    print("Found checkpoint:", resume_ckpt)
    try:
        print("Attempting resume=True from checkpoint...")
        model = YOLO(resume_ckpt)
        model.train(resume=True)
    except Exception as exc:
        print("Resume failed (often means prior run already finished).")
        print("Falling back to new run initialized from checkpoint.")
        print("Error:", exc)
        model = YOLO(resume_ckpt)
        model.train(resume=False, **train_args)
else:
    print("No previous checkpoint found. Starting fresh from base model.")
    model = YOLO(f"{MODEL_NAME}.pt")
    model.train(resume=False, **train_args)


## 4) Locate Latest Run and Export Artifacts

This cell:
- finds latest run folder
- reads `results.csv` to append metrics to exported filename when available
- exports `best.pt`, `last.pt`, and zipped run folder to `DRIVE_EXPORT_DIR`


In [ ]:
run_candidates = glob.glob(os.path.join(PROJECT_RUNS_DIR, "*", "weights", "best.pt"))
if not run_candidates:
    raise FileNotFoundError(f"No best.pt found under {PROJECT_RUNS_DIR}")

best_ckpt = max(run_candidates, key=os.path.getmtime)
latest_run_dir = os.path.dirname(os.path.dirname(best_ckpt))
last_ckpt = os.path.join(latest_run_dir, "weights", "last.pt")
results_csv = os.path.join(latest_run_dir, "results.csv")

metric_suffix = ""
if os.path.exists(results_csv):
    df = pd.read_csv(results_csv)
    df.columns = [c.strip() for c in df.columns]

    map50_col = next((c for c in ["metrics/mAP50(P)", "metrics/mAP50(M)", "metrics/mAP50(B)"] if c in df.columns), None)
    map5095_col = next((c for c in ["metrics/mAP50-95(P)", "metrics/mAP50-95(M)", "metrics/mAP50-95(B)"] if c in df.columns), None)

    if map50_col and map5095_col:
        idx = df[map5095_col].idxmax()
        m50 = float(df.loc[idx, map50_col])
        m5095 = float(df.loc[idx, map5095_col])
        metric_suffix = f"-mAP5095_{m5095:.4f}-mAP50_{m50:.4f}"
        print(f"Best metrics: {map5095_col}={m5095:.4f}, {map50_col}={m50:.4f}")

stamp = datetime.now().strftime("%Y-%m-%d-%H%M%S")
export_base = f"{PROJECT_NAME}-{stamp}-{DATASET_NAME}{metric_suffix}"

export_best = os.path.join(DRIVE_EXPORT_DIR, f"{export_base}-best.pt")
export_last = os.path.join(DRIVE_EXPORT_DIR, f"{export_base}-last.pt")
export_zip = os.path.join(DRIVE_EXPORT_DIR, f"{export_base}-run.zip")

shutil.copy2(best_ckpt, export_best)
if os.path.exists(last_ckpt):
    shutil.copy2(last_ckpt, export_last)

zip_tmp_base = f"/tmp/{export_base}"
shutil.make_archive(zip_tmp_base, "zip", latest_run_dir)
shutil.copy2(f"{zip_tmp_base}.zip", export_zip)

print("Latest run dir:", latest_run_dir)
print("Exported best:", export_best)
print("Exported last:", export_last if os.path.exists(last_ckpt) else "last.pt missing")
print("Exported zip:", export_zip)


## 5) Validate and Predict Using Dynamic Paths

No hardcoded run names are used; this always uses the latest `best.pt`.


In [ ]:
# Resolve validation image directory from data.yaml
with open(YAML_PATH, "r", encoding="utf-8") as f:
    data_cfg = yaml.safe_load(f)

val_rel_or_abs = data_cfg.get("val")
if val_rel_or_abs is None:
    raise ValueError("'val' split not found in data.yaml")

val_images_dir = val_rel_or_abs if os.path.isabs(val_rel_or_abs) else os.path.join(DATASET_DIR, val_rel_or_abs)
if not os.path.exists(val_images_dir):
    raise FileNotFoundError(f"Validation images directory not found: {val_images_dir}")

print("Running validation with:", best_ckpt)
val_model = YOLO(best_ckpt)
val_model.val(data=YAML_PATH, imgsz=IMG_SIZE, batch=BATCH_SIZE, device=DEVICE)

pred_name = f"predict_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
print("Running prediction on:", val_images_dir)
val_model.predict(
    source=val_images_dir,
    imgsz=IMG_SIZE,
    conf=CONF_PRED,
    device=DEVICE,
    save=True,
    project=PROJECT_RUNS_DIR,
    name=pred_name,
)

pred_dir = os.path.join(PROJECT_RUNS_DIR, pred_name)
pred_imgs = glob.glob(os.path.join(pred_dir, "*.jpg")) + glob.glob(os.path.join(pred_dir, "*.png"))

print("Prediction output:", pred_dir)
print("Images found:", len(pred_imgs))
for img_path in pred_imgs[:3]:
    display(Image(filename=img_path, width=500))


## Notes

- For best resume reliability, keep `RUNS_ROOT` on Drive as configured.
- If training is interrupted, re-run from section 3 onward.
- If you switch datasets/models, update section 1 only.
